# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

### Libraries

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
from pathlib import Path
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

### Input

Change these variables to what you need. The locations must be a string with each continent separated by a comma and space. The serotypes and genotypes must be lists. The dates must be strings. 

In [2]:
# Dates and locations
locations = "Antarctica, North America, South America"
serotypes = ["H5N1"]
genotypes = ["B3.13"]
start_date = "11-01-2021"
end_date = "07-17-2026"

# Make a date range
date_range = start_date + "--" + end_date


### Paths

Make sure these paths fit your schema. Here is the general tree structure: </br>

* Avian Flu: This is where the code in this repository is housed. "references" is a subdirectory here.
* Avian Flu Files: This is a sister directory to the repository -- that is, the code repository and the input/output files both have the same parent directory, which is the "home" variable.
* Avian Flu Files/NCBI Virus: This is the NCBI Virus directory. There are three subdirectories here: "downloads", "temp", and "complete".
* NCBI Virus/downloads: input data
* NCBI Virus/temp: intermediate data created between input and output. A subdirectory is created for this specific date range.
* NCBI Virus/complete: output data

<img src="../Avian_Flu_Files/Presentations/ncbi_virus_file_tree.png" width="500" height="250" alt="NCBI Virus file tree structure">

In [3]:
# Paths

home = Path.home() / "OneDrive - National Institutes of Health/Documents/Virus_Evolution/"
avian_flu_files = home / "Avian_Flu_Files" # Avian flu files is a sister directory of the directory this code is housed in
references = home / "Avian_Flu" / "references"
downloads = Path.home() / "Downloads"

downloads_saved = avian_flu_files / "NCBI_Virus/downloads" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))  

temp_files = avian_flu_files / "NCBI_Virus/temp" / ("segments_" + date_range)
if not temp_files.exists():
    Path.mkdir(temp_files, parents=True, exist_ok=True)
complete_files = avian_flu_files / "NCBI_Virus/complete" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))
if not complete_files.exists(): # checking if the directory exists or not
    Path.mkdir(complete_files, parents=True, exist_ok=True) # if the directory is not present then create it

states_ref = pd.read_csv(references / "states_ref.csv")

## Downloading Data

In [4]:
os.chdir(downloads)

if not downloads_saved.exists(): # checking if the directory exists or not
    Path.mkdir(downloads_saved, parents=True, exist_ok=True) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0: # If we have downloaded files already, skip
        break 
    else: # If we don't have any downloaded files, get files
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [5]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Assembly, as_index=False).size()
print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Assembly")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_counted.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments = metadata_complete_segs
metadata_segments

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_29176\3687857257.py:3: DtypeWarning: Columns (3,4,5,18) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv("sequences.csv")


164081
              Assembly  size
0      GCA_038929245.1     8
1      GCA_038932225.1     8
2      GCA_038932275.1     8
3      GCA_038932295.1     8
4      GCA_038933705.1     8
...                ...   ...
18776  GCA_059516965.1     8
18777  GCA_059516985.1     8
18778  GCA_059516995.1     8
18779  GCA_059517005.1     8
18780  GCA_059517015.1     8

[18781 rows x 2 columns]


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PZ686517.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Meleagris gallopavo,"swab, lung","Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-09-16,2026-07-14,ssRNA(-),8
1,PZ686518.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Meleagris gallopavo,"swab, lung","Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-09-16,2026-07-14,ssRNA(-),8
2,PZ686519.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Meleagris gallopavo,"swab, lung","Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-09-16,2026-07-14,ssRNA(-),8
3,PZ686520.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Meleagris gallopavo,"swab, lung","Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-09-16,2026-07-14,ssRNA(-),8
4,PZ686521.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Meleagris gallopavo,"swab, lung","Lantz,K., Stuber,T.",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-09-16,2026-07-14,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150227,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
150228,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
150229,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
150230,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [6]:
os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Double-check the de-duplication
print(len(sequences_fasta)) 
# print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PZ699977.1 |Influenza A virus (A/CATTLE/ID/26...
1         >PZ699978.1 |Influenza A virus (A/CATTLE/ID/26...
2         >PZ699979.1 |Influenza A virus (A/CATTLE/ID/26...
3         >PZ699980.1 |Influenza A virus (A/CATTLE/ID/26...
4         >PZ699981.1 |Influenza A virus (A/CATTLE/ID/26...
                                ...                        
164076    >OK205883.1 |Influenza A virus (A/chicken/Vera...
164077    >OK205884.1 |Influenza A virus (A/chicken/Vera...
164078    >OK205885.1 |Influenza A virus (A/chicken/Vera...
164079    >OK205886.1 |Influenza A virus (A/chicken/Vera...
164080    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 164081, dtype: object
164081
148720


In [17]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,Partial_Header_temp,Partial_Header,Partial_Header_Merge
0,PZ686517.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686517.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
1,PZ686518.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686518.1 |Influenza A virus (A/Turkey/CA/24...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
2,PZ686519.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686519.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
3,PZ686520.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686520.1 |Influenza A virus (A/Turkey/CA/24...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
4,PZ686521.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686521.1 |Influenza A virus (A/Turkey/CA/24...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148715,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...,1864,>1864,>1864
148716,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...,1864,>1864,>1864
148717,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...,1864,>1864,>1864
148718,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,1864,>1864,>1864


## Find genotypes

Set up files that are friendly to multi-genoflu, then run multi-genoflu.

In [18]:
metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

print(len(metadata_segments))

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_29176\2535715611.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_29176\2535715611.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)


148712


### Create FASTA files of unknown genotypes 

In [19]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,Partial_Header_temp,Partial_Header,Partial_Header_Merge
0,PZ686517.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686517.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
1,PZ686518.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686518.1 |Influenza A virus (A/Turkey/CA/24...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
2,PZ686519.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686519.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
3,PZ686520.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686520.1 |Influenza A virus (A/Turkey/CA/24...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
4,PZ686521.1,GenBank,GCA_059516225.1,SRR30811263,SAMN43929866,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2024-09-16,2026-07-14,ssRNA(-),8,>PZ686521.1 |Influenza A virus (A/Turkey/CA/24...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024,>A_Turkey_CA_24_027086_001_original_2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148715,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...,1864,>1864,>1864
148716,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...,1864,>1864,>1864
148717,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...,1864,>1864,>1864
148718,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,1864,>1864,>1864


In [20]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments["Partial_Header"] = metadata_segments["Partial_Header_temp"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name
metadata_segments = metadata_segments.dropna(subset="Partial_Header")

print(metadata_segments["Partial_Header"].values[0:5])

['>A_Turkey_CA_24_027086_001_original_2024'
 '>A_Turkey_CA_24_027086_001_original_2024'
 '>A_Turkey_CA_24_027086_001_original_2024'
 '>A_Turkey_CA_24_027086_001_original_2024'
 '>A_Turkey_CA_24_027086_001_original_2024']


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_29176\3389490406.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header"] = metadata_segments["Partial_Header_temp"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name


In [11]:
# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

0         >A_Turkey_CA_24_027086_001_original_2024
1         >A_Turkey_CA_24_027086_001_original_2024
2         >A_Turkey_CA_24_027086_001_original_2024
3         >A_Turkey_CA_24_027086_001_original_2024
4         >A_Turkey_CA_24_027086_001_original_2024
                            ...                   
148715                                       >1864
148716                                       >1864
148717                                       >1864
148718                                       >1864
148719                                       >1864
Name: Partial_Header, Length: 148712, dtype: object
['>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horned_Owl_CA_26G01049_002_original_2026'
 '>A_Great_Horne

### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To avoid job kill:
```
sinteractive
```
To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [12]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [29]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# Merge
metadata_genoflu = metadata_segments.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 
metadata_genoflu = metadata_genoflu.rename(columns={"Genotype_y":"Genotype", "Genotype_x":"Serotype"})


# # Fill the rest of the 8 segments with the same genotype
# metadata_genoflu = metadata_genoflu.ffill(limit_area="inside", limit=7)

print(metadata_genoflu)


         Accession GenBank_RefSeq         Assembly SRA_Accession  \
0       PZ686517.1        GenBank  GCA_059516225.1   SRR30811263   
1       PZ686518.1        GenBank  GCA_059516225.1   SRR30811263   
2       PZ686519.1        GenBank  GCA_059516225.1   SRR30811263   
3       PZ686520.1        GenBank  GCA_059516225.1   SRR30811263   
4       PZ686521.1        GenBank  GCA_059516225.1   SRR30811263   
...            ...            ...              ...           ...   
148643  OK205699.1        GenBank  GCA_039174385.1           NaN   
148644  OK205700.1        GenBank  GCA_039174385.1           NaN   
148645  OK205701.1        GenBank  GCA_039174385.1           NaN   
148646  OK205702.1        GenBank  GCA_039174385.1           NaN   
148647  OK205703.1        GenBank  GCA_039174385.1           NaN   

           BioSample    BioProject      Organism_Name  \
0       SAMN43929866  PRJNA1102327  Influenza A virus   
1       SAMN43929866  PRJNA1102327  Influenza A virus   
2       SAMN

In [30]:
# Cut down to only columns we want
metadata_genoflu = metadata_genoflu[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "Partial_Header_Merge"]] #, "Strain"]]

# Get genbank strain name
 
metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu = metadata_genoflu.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

In [ ]:
metadata_genoflu

# Make sure we only have the serotype(s) we want -- this code only works for a single serotype, so change it if multiple are ever needed
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,Partial_Header_Merge,genbank_name
0,PZ686517.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686517.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,H5N1,1,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
1,PZ686518.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686518.1 |Influenza A virus (A/Turkey/CA/24...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,H5N1,2,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
2,PZ686519.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686519.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,H5N1,3,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
3,PZ686520.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686520.1 |Influenza A virus (A/Turkey/CA/24...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
4,PZ686521.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686521.1 |Influenza A virus (A/Turkey/CA/24...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147539,OP221403.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not assigned: Only 5 segments >98.0% match fou...,USA,>OP221403.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
147540,OP221404.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not assigned: Only 5 segments >98.0% match fou...,USA,>OP221404.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAGC...,H5N1,1,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
147541,OP222196.1,GCA_039271695.1,Influenza A virus (A/bald eagle/Florida/W22-19...,bald eagle,2022-03-08,NaN,A/bald eagle/Florida/W22-191/2022,Not assigned: Only 7 segments >98.0% match fou...,USA,>OP222196.1 |Influenza A virus (A/bald eagle/F...,AGCGAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_191_2022,A/bald eagle/Florida/W22-191/2022
147542,OP222197.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not assigned: Only 5 segments >98.0% match fou...,USA,>OP222197.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022


## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

After running the below code, **STOP TO CHECK** if any new animals appear

In [36]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, write code dealing with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv", index=False) # Make sure name is different to avoid overwriting the first reference 


['rock pigeon', 'white-faced ibis', 'franklin gull', 'hooded merganser', 'cackling goose', 'northwestern crow', 'blue-winged teal', 'western sandpiper', 'striped skunk', 'red-necked phalarope', 'blackbird', 'black-necked swan', 'black swift', 'wild bird', 'procellaria aequinoctialis', 'serval', 'layer chicken', 'glaucous gull', 'quail', 'goose', 'pig', 'double-crested cormorant', 'skunk', 'american crow', 'golden eagle', 'magpie', 'partridge', 'duck', 'ohio', 'greater white-fronted goose', 'royal tern', 'penguin', 'pintail duck', 'eurasian collared dove', 'black swan', 'otaria flavescens', 'thalasseus maximus', 'chukar', 'waterfowl', 'emu', 'osprey', 'common tern', 'eagle', 'bluejay', 'pelecanus', 'common eiders', 'ring-necked duck', 'white-winged dove', 'black vulture', 'furnarius rufus', 'guineafowl', 'american white pelican', 'american widgeon', 'sharp-shinned hawk', 'greater scaup', 'poultry', 'glaucous-winged gull', 'megascops choliba', 'ostrich', 'band-tailed gull', 'snowy owl', 

In [34]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [37]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype"] = metadata_genoflu["Genotype"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
# metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [38]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,Partial_Header_Merge,genbank_name
0,PZ686517.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686517.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,H5N1,1,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
1,PZ686518.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686518.1 |Influenza A virus (A/Turkey/CA/24...,ATGGATGTCAATCCGACCTTACTCTTCTTGAAAGTTCCAGCGCAAA...,H5N1,2,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
2,PZ686519.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686519.1 |Influenza A virus (A/Turkey/CA/24...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,H5N1,3,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
3,PZ686520.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686520.1 |Influenza A virus (A/Turkey/CA/24...,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
4,PZ686521.1,GCA_059516225.1,Influenza A virus (A/Turkey/CA/24-027086-001-o...,Turkey,2024-09-16,SRR30811263,A/Turkey/CA/24-027086-001-original/2024,B3.13,USA: CA,>PZ686521.1 |Influenza A virus (A/Turkey/CA/24...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5,A_Turkey_CA_24_027086_001_original_2024,A/Turkey/CA/24-027086-001-original/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147539,OP221403.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP221403.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
147540,OP221404.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP221404.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAGC...,H5N1,1,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
147541,OP222196.1,GCA_039271695.1,Influenza A virus (A/bald eagle/Florida/W22-19...,bald eagle,2022-03-08,NaN,A/bald eagle/Florida/W22-191/2022,Not,USA,>OP222196.1 |Influenza A virus (A/bald eagle/F...,AGCGAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_191_2022,A/bald eagle/Florida/W22-191/2022
147542,OP222197.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP222197.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022


In [ ]:
def geo_location_normalize(geolocation: str) -> str:

    geolocation = geolocation.replace(":", ",") # We will need to split on commas later; e.g. USA: MD -> USA, MD

    country = geolocation.split(",")[0] # Get the first part of the geolocation, aka the country 

    state = geolocation.split(",")[-1] # Get the last part of the geolocation, aka the state

    state = state.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country = country.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country_state = country + "-" + state # Log needed

    return country_state 

# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

      
    metadata["Geo_Location_State_USA"] = metadata["Geo_Location_State_med"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name
    else x)
    

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_USA"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    return metadata

def geo_location_strain_name(metadata, state_ref_file):
    # If there is no state in the metadata, try the strain name
    state_ref = pd.read_csv(state_ref_file)

    metadata["strain_name_state_nonhuman"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2])

    metadata["strain_name_state_human"] = metadata["genbank_name"].apply(lambda x: x.split("/")[1])

    # If nonhuman, do strain name [2]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] != "human"), metadata["strain_name_state_nonhuman"], metadata["Geo_Location_State_USA"])

    # If human, do strain name [1]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] == "human"), metadata["strain_name_state_human"], metadata["Geo_Location_State_USA"])

    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name"].apply(geo_location_normalize)

    metadata["Geo_Location_State_Strain_Name_State"] = metadata["Geo_Location_State_Strain_Name"].apply(lambda x: x.split("-")[-1])
    
    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name_State"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name, and is just USA
    else ""
    if x == "USA" or x == "United_States"
    
    else x)

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_Strain_Name"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata

    
    return metadata

In [45]:
print(states_ref[states_ref['State'].str.contains("New_York")]["Abbreviation"].values[0])

NY


In [ ]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))



In [46]:
os.chdir(references)
metadata_genoflu = geo_location_get(metadata_genoflu, "states_ref.csv")

In [47]:
metadata_genoflu = geo_location_strain_name(metadata_genoflu, "states_ref.csv")

In [ ]:
# os.chdir(downloads_saved)
# metadata_genoflu.to_csv("metadata_checkpoint.csv")

In [ ]:
# If there is no SRA Accession, replace identifier with Assembly -- this part might be duplicating accessions :(
metadata_genoflu["Identifier"] = metadata_genoflu["Assembly"].apply(lambda x: x if metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0] == "" else metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0]) # np.where(metadata_genoflu['SRA_Accession'] != "", metadata_genoflu['SRA_Accession'], metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]))
# # If there is no Assembly, replace identifier with Accession -- only for PB2
# metadata_genoflu["Identifier"] = np.where(metadata_genoflu["SRA_Accession_intermediate"] == "", metadata_genoflu["Accession"].apply(lambda x: metadata_genoflu[metadata_genoflu["Accession"] == x]["Accession"].values[0] if metadata_genoflu[metadata_genoflu["Accession"] == x].loc[:, "Segment"].values[0] == 1 else np.nan), metadata_genoflu["SRA_Accession_intermediate"])
# # Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
# metadata_genoflu.loc[:,"Identifier"] = metadata_genoflu.loc[:, "Identifier"].ffill(limit=7, limit_area="inside")

print((metadata_genoflu[metadata_genoflu["Identifier"].str.contains("SRR")])) # metadata_genoflu[(metadata_genoflu["Identifier"].str.contains("GCA")) | 

In [ ]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_New"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [ ]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
# metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


81504
81472


## Rename segments and make complete FASTA files

In [ ]:
# Set up segments

# if len(genotypes) > 3: # If we're not doing maintenance only
#     genotypes.append("Not assigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.13_PB2
B3.13_PB1
B3.13_PA
B3.13_HA
B3.13_NP
B3.13_NA
B3.13_MP
B3.13_NS


In [ ]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

        Accession         Assembly  \
1104   PZ479596.1  GCA_058019015.1   
1112   PZ479604.1  GCA_058019025.1   
1120   PZ479612.1  GCA_058019055.1   
1128   PZ479620.1  GCA_058019105.1   
1136   PZ479628.1  GCA_058019115.1   
...           ...              ...   
81415  PP756076.1  GCA_039465795.1   
81423  PP756084.1  GCA_039464765.1   
81431  PP756092.1  GCA_039465935.1   
81439  PP756100.1  GCA_039466035.1   
81447  PP740729.1  GCA_039251605.1   

                                           GenBank_Title       Host  \
1104   Influenza A virus (A/cattle/ID/26G07162-001-or...     cattle   
1112   Influenza A virus (A/cattle/ID/26G07162-002-or...     cattle   
1120   Influenza A virus (A/cattle/ID/26G07162-003-or...     cattle   
1128   Influenza A virus (A/cattle/ID/26G07162-004-or...     cattle   
1136   Influenza A virus (A/cattle/ID/26G07162-005-or...     cattle   
...                                                  ...        ...   
81415  Influenza A virus (A/cattle/New Mexico/